<a href="https://colab.research.google.com/github/google-ai-edge/LiteRT-CLI/blob/main/examples/litert_cli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 The LiteRT CLI Authors.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

# LiteRT CLI Demo

This notebook demonstrates how to use the `litert-cli` tool to convert a PyTorch model, quantize it and run it.

## 🛠️ 1. Environment Setup & Installation

In [ ]:
# Update protobuf version
!pip install -U protobuf
# Install required packages
!pip install torch torchvision
# Default torchao is 
!pip install -U torchao

# Install litert-cli
!pip install litert-cli-nightly

# 'litert compile' depends on Clang, and below make sure your Clang has version `18.x.x` or above
!wget https://apt.llvm.org/llvm.sh
!chmod +x llvm.sh
!sudo ./llvm.sh 18 all

## 📝 2. Prepare PyTorch Model Script

In [ ]:
%%writefile resnet18.py
import torch
import torchvision

def get_model(batch_size: int = 1) -> torch.nn.Module:
  model = torchvision.models.resnet18(
      weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1
  )
  model.eval()
  return model

def get_args(batch_size: int = 1) -> tuple[torch.Tensor, ...]:
  return (torch.randn(batch_size, 3, 224, 224),)

## 🔄 3. Model Conversion (PyTorch -> LiteRT)

In [ ]:
# Convert PyTorch ResNet18 model to LiteRT
!litert convert resnet18.py --output resnet18

## 📉 4. Model Quantization

In [ ]:
# Quantize the ResNet18 model
!litert quantize resnet18/resnet18.tflite --recipe dynamic_wi8_afp32 --output resnet18/resnet18_int8_dynamic.tflite
!litert quantize resnet18/resnet18.tflite --recipe weight_only_wi8_afp32 --output resnet18/resnet18_int8_weight_only.tflite

## 🚀 5. Run Inference
### 🖥️ 5.1 CPU Inference

In [ ]:
# Run Inference on Desktop (Colab VM CPU)
!litert run resnet18/resnet18.tflite
!litert run resnet18/resnet18_int8_dynamic.tflite  --desktop --cpu --iterations 1

# Download EfficientNet-B1 model directly from HuggingFace (No conversion needed)
!litert download litert-community/efficientnet_b1 --file "*.tflite" --output efficientnet

# Run Inference for EfficientNet on Desktop CPU
!litert run efficientnet/efficientnet_b1.tflite

### 🎮 5.2 GPU Inference

In [ ]:
# Run Inference on Desktop with GPU (Colab VM GPU), and please make you connect to a machine
# with GPU. You can click "change runtime type" to find options there.
!litert run resnet18/resnet18.tflite
!litert run resnet18/resnet18_int8_dynamic.tflite --desktop --gpu --iterations 1 --print-tensors

# Run Inference for EfficientNet on Desktop GPU
!litert run efficientnet/efficientnet_b1.tflite --gpu

## ⚙️ 6. NPU Offline Compilation (AOT)

In [ ]:
# Compile the model for qualcomm NPU: SM8750
# TIP: Only support running on Linux and it might take a few minutes, given download large SDKs from SOCs.
#
!litert compile resnet18/resnet18.tflite --target sm8750

# Compile EfficientNet model for Qualcomm NPU: SM8750
!litert compile efficientnet/efficientnet_b1.tflite --target sm8750

## 🏁 7. Benchmark in Google AI Edge Portal

In [ ]:
# @title Login to Google Cloud & Configure GCP Project
# @markdown Please login into Google Cloud and make sure you have joined the EAP for Google AI Edge Portal. Check: https://ai.google.dev/edge/ai-edge-portal
# @markdown ---
# @markdown **GCP Configuration**
# @markdown Please specify your GCP Project ID. If `gcp_bucket` is left blank, a default bucket `<gcp_project>-litert-models` will be used.
gcp_project = "your-own-gcp-project-id" # @param {type:"string"}
gcp_bucket = "" # @param {type:"string"}

if not gcp_project or gcp_project == "your-own-gcp-project-id":
  raise ValueError("Error: Please specify a valid GCP Project ID in the form above.")

if not gcp_bucket:
  gcp_bucket = f"{gcp_project}-litert-models"

print(f"Configured GCP Project: {gcp_project}")
print(f"Configured GCS Bucket: gs://{gcp_bucket}")

# Install gcloud and login
!pip install gcloud
!gcloud auth login

# Benchmark ResNet18 on Google AI Edge Portal
!litert benchmark resnet18/resnet18.tflite --gcp --device "pixel 7" --cpu --gcp-project {gcp_project}
!litert benchmark resnet18/resnet18.tflite --gcp --devices "pixel 7, sm-s931u1" --gpu --gcp-project {gcp_project} --gcp-bucket {gcp_bucket}

# Benchmark EfficientNet on Google AI Edge Portal with CPU and NPU
!litert benchmark efficientnet/efficientnet_b1.tflite --gcp --device "pixel 7" --cpu --gcp-project {gcp_project}
!litert benchmark efficientnet/efficientnet_b1.tflite --gcp --device "sm-s931u1" --npu --soc-model SM8750 --gcp-project {gcp_project} --gcp-bucket {gcp_bucket}

## Advanced Usage

### Android Deployment
To run or benchmark on Android, you need an ADB connected device. These commands typically look like:
```bash
# Run on Android
!litert run resnet18/resnet18.tflite --android --cpu

# Benchmark on Android
!litert benchmark resnet18/resnet18.tflite --android
```
These are not executed in this notebook as Colab does not have access to a physical Android device by default.